# NBA Injury Prediction — Preliminary Model (Random Forest)
**CSC 525 — Team 9**

This notebook:
1. Downloads both datasets via the Kaggle API
2. Joins them into a single player-season dataset with a binary injury label
3. Trains a Random Forest classifier as the preliminary model
4. Evaluates using AUC-ROC, Precision, Recall, F1, and feature importances

**Prerequisite:** Kaggle API credentials must be configured at `~/.kaggle/kaggle.json`.  
Get your token from: https://www.kaggle.com/settings → API → Create New Token

## Section 0 — Setup & Configuration

In [ ]:
# !pip install kaggle pandas scikit-learn matplotlib seaborn numpy

In [ ]:
import os
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

# ── Configuration ────────────────────────────────────────────────────────────
DATA_DIR      = "data"
INJURY_CSV    = os.path.join(DATA_DIR, "injuries_2010-2020.csv")   # updated after download
PLAYERS_CSV   = os.path.join(DATA_DIR, "all_seasons.csv")
RANDOM_STATE  = 42
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(DATA_DIR, exist_ok=True)
print("Setup complete.")

## Section 1 — Download Datasets via Kaggle API

In [ ]:
import kaggle

# Dataset 1: NBA Injury Stats 1951-2023
print("Downloading injury dataset...")
kaggle.api.dataset_download_files(
    "loganlauton/nba-injury-stats-1951-2023",
    path=DATA_DIR,
    unzip=True,
    quiet=False
)

# Dataset 2: NBA Players Biometric + Box Score
print("\nDownloading players dataset...")
kaggle.api.dataset_download_files(
    "justinas/nba-players-data",
    path=DATA_DIR,
    unzip=True,
    quiet=False
)

# Show what was downloaded
print("\nFiles in data/:")
for f in sorted(os.listdir(DATA_DIR)):
    print(f"  {f}")

In [ ]:
# ── Identify correct filenames after extraction ───────────────────────────────
# The injury dataset may extract as 'injuries_2010-2020.csv' or similar.
# Update INJURY_CSV below if the filename differs.
csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith(".csv")]
print("CSV files found:", csv_files)

# Auto-detect injury CSV (contains 'injur' in name)
injury_candidates = [f for f in csv_files if "injur" in f.lower()]
player_candidates = [f for f in csv_files if "season" in f.lower() or "player" in f.lower()]

print("Injury CSV candidates:", injury_candidates)
print("Player CSV candidates:", player_candidates)

# Set paths — adjust if auto-detection picks wrong file
INJURY_CSV  = os.path.join(DATA_DIR, injury_candidates[0])
PLAYERS_CSV = os.path.join(DATA_DIR, player_candidates[0])
print(f"\nUsing injury CSV:  {INJURY_CSV}")
print(f"Using players CSV: {PLAYERS_CSV}")

## Section 2 — Data Loading

In [ ]:
injuries_raw = pd.read_csv(INJURY_CSV)
players_raw  = pd.read_csv(PLAYERS_CSV)

print("=== Injury dataset ===")
print(f"Shape: {injuries_raw.shape}")
print(injuries_raw.dtypes)
display(injuries_raw.head(3))

print("\n=== Players dataset ===")
print(f"Shape: {players_raw.shape}")
print(players_raw.dtypes)
display(players_raw.head(3))

In [ ]:
# Identify the date column in the injury dataset
# Common names: 'Date', 'date', 'Acquired', 'Relinquished'
print("Injury dataset columns:", injuries_raw.columns.tolist())
print("\nPlayers dataset columns:", players_raw.columns.tolist())

# Check the season format in the players dataset
if "season" in players_raw.columns:
    print("\nSample season values:", players_raw["season"].unique()[:5].tolist())

## Section 3 — Dataset Joining

This is the core data engineering step. We:
1. Parse injury dates → season year
2. Normalize player names in both datasets (handle accents, suffixes)
3. Convert season year integer to `"YYYY-YY"` string format
4. Aggregate injuries to a binary label per (player, season)
5. Left-join the players dataset with the injury labels

### 3a — Detect Date Column & Parse Season Year

In [ ]:
# Detect which column holds the date in the injury dataset
DATE_COL = None
for candidate in ["Date", "date", "DATE"]:
    if candidate in injuries_raw.columns:
        DATE_COL = candidate
        break

if DATE_COL is None:
    raise ValueError(f"No date column found. Columns: {injuries_raw.columns.tolist()}")

print(f"Using date column: '{DATE_COL}'")
print("Sample values:", injuries_raw[DATE_COL].head(5).tolist())

# Parse dates; coerce errors to NaT
injuries_raw["date_parsed"] = pd.to_datetime(injuries_raw[DATE_COL], errors="coerce")

n_before = len(injuries_raw)
injuries_raw = injuries_raw.dropna(subset=["date_parsed"])
print(f"Dropped {n_before - len(injuries_raw)} rows with unparseable dates")
print(f"Date range: {injuries_raw['date_parsed'].min()} to {injuries_raw['date_parsed'].max()}")

In [ ]:
# NBA season labeled "2019-20" runs Oct 2019 – Jun 2020
# Convention: season start year is the key
#   Oct–Dec of year Y  → season Y
#   Jan–Sep of year Y  → season Y-1
def date_to_season_year(dt):
    return dt.year if dt.month >= 10 else dt.year - 1

injuries_raw["season_year"] = injuries_raw["date_parsed"].apply(date_to_season_year)
print("Season year range:", injuries_raw["season_year"].min(), "to", injuries_raw["season_year"].max())

### 3b — Normalize Player Names

In [ ]:
def normalize_name(name):
    """Lowercase, strip whitespace, remove common suffixes, convert to ASCII."""
    if pd.isna(name):
        return ""
    name = str(name).lower().strip()
    name = " ".join(name.split())  # collapse internal whitespace
    # Remove name suffixes that appear inconsistently across datasets
    for suffix in [" jr.", " jr", " sr.", " sr", " ii", " iii", " iv"]:
        if name.endswith(suffix):
            name = name[: -len(suffix)].strip()
    # Convert accented characters to ASCII (e.g. Dončić → Doncic)
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    return name

# Detect player name column in injury dataset
PLAYER_COL_INJ = None
for candidate in ["Player", "player", "PLAYER", "Relinquished", "Name"]:
    if candidate in injuries_raw.columns:
        PLAYER_COL_INJ = candidate
        break

if PLAYER_COL_INJ is None:
    raise ValueError(f"No player name column found in injury dataset. Columns: {injuries_raw.columns.tolist()}")

print(f"Using player column in injury dataset: '{PLAYER_COL_INJ}'")

injuries_raw["player_norm"] = injuries_raw[PLAYER_COL_INJ].apply(normalize_name)
players_raw["player_norm"]  = players_raw["player_name"].apply(normalize_name)

print("Sample normalized names (injuries):", injuries_raw["player_norm"].head(5).tolist())
print("Sample normalized names (players):",  players_raw["player_norm"].head(5).tolist())

### 3c — Convert Season Year to Season String

In [ ]:
# The justinas players dataset uses "2019-20" format
# Convert injury dataset's integer year to the same format
def year_to_season_str(y):
    return f"{y}-{str(y + 1)[-2:]}"  # 2019 → "2019-20"

injuries_raw["season"] = injuries_raw["season_year"].apply(year_to_season_str)
print("Sample season strings:", injuries_raw["season"].unique()[:5].tolist())
print("Players dataset seasons:", sorted(players_raw["season"].unique())[:5])

### 3d — Aggregate to Binary Injury Label

In [ ]:
# A player with multiple injuries in a season is still label=1
injury_labels = (
    injuries_raw
    .groupby(["player_norm", "season"])
    .size()
    .reset_index(name="injury_count")
    .assign(injured=1)[["player_norm", "season", "injured"]]
)

print(f"Unique (player, season) injury records: {len(injury_labels)}")
print(f"Unique injured players: {injury_labels['player_norm'].nunique()}")
display(injury_labels.head())

### 3e — Merge Datasets

In [ ]:
# Left join: keep all player-seasons; add injury flag where available
merged = players_raw.merge(
    injury_labels,
    on=["player_norm", "season"],
    how="left"
)

# Players with no injury record → not injured (0)
merged["injured"] = merged["injured"].fillna(0).astype(int)

print(f"Players dataset rows:  {len(players_raw):,}")
print(f"Merged dataset rows:   {len(merged):,}")
print(f"Injury rate:           {merged['injured'].mean():.3f} ({merged['injured'].sum():,} injured seasons)")
print(f"Season range:          {merged['season'].min()} to {merged['season'].max()}")

# How many player-seasons matched to an injury record?
matched = merged["injured"].sum()
print(f"\nInjured player-seasons matched from injury dataset: {matched:,}")
print(f"Injury label dataset size:                          {len(injury_labels):,}")
print(f"Match rate:                                         {matched/len(injury_labels):.1%}")

In [ ]:
# ── Final Joined Dataset Overview ────────────────────────────────────────────
print(f"Shape: {merged.shape}  ({merged.shape[0]:,} player-seasons × {merged.shape[1]} columns)")
print(f"Seasons covered:  {merged['season'].min()} → {merged['season'].max()}")
print(f"Unique players:   {merged['player_norm'].nunique():,}")
print(f"Injured seasons:  {merged['injured'].sum():,}  ({merged['injured'].mean():.1%} of all player-seasons)")
print()

# Display key columns only for readability
key_cols = ["player_name", "season", "age", "player_height", "player_weight",
            "gp", "pts", "reb", "ast", "usg_pct", "ts_pct", "injured"]
key_cols = [c for c in key_cols if c in merged.columns]

print("=== Sample rows — Injured (injured=1) ===")
display(merged[merged["injured"] == 1][key_cols].head(5))

print("=== Sample rows — Not Injured (injured=0) ===")
display(merged[merged["injured"] == 0][key_cols].head(5))

print("\n=== All columns in the merged dataset ===")
print(merged.columns.tolist())

### 3f — View of the Final Joined Dataset

**What was joined and how:**

| | Injury Dataset | Players Dataset |
|---|---|---|
| **Source** | `loganlauton/nba-injury-stats-1951-2023` | `justinas/nba-players-data` |
| **Grain** | One row per injury event | One row per player per season |
| **Join key 1** | `player_norm` — name lowercased, accents stripped, suffixes (Jr./Sr./III) removed | `player_norm` — same normalization applied |
| **Join key 2** | `season` derived from injury date (e.g. 2019-11-03 → `"2019-20"`) | `season` column already in `"2019-20"` format |
| **Join type** | **Left join** on players dataset — every player-season is kept; `injured=0` if no match found |
| **Label** | Aggregated: ≥1 injury event in a season → `injured=1`, else `injured=0` |

## Section 4 — Exploratory Data Analysis

### 4a — Class Balance

In [ ]:
counts = merged["injured"].value_counts()
fig, ax = plt.subplots(figsize=(5, 4))
counts.plot(kind="bar", ax=ax, color=["steelblue", "tomato"], edgecolor="white")
ax.set_xticklabels(["Not Injured (0)", "Injured (1)"], rotation=0)
ax.set_title("Class Distribution")
ax.set_ylabel("Count")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + 0.3, p.get_height() + 20), ha="center")
plt.tight_layout()
plt.show()

print(merged["injured"].value_counts(normalize=True).rename({0: "Not Injured", 1: "Injured"}).to_string())

### 4b — Missing Values

In [ ]:
missing = merged.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(merged) * 100).round(1)
missing_df = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
display(missing_df[missing_df["missing_count"] > 0])

### 4c — Feature Distributions by Injury Label

In [ ]:
numeric_candidates = ["age", "player_height", "player_weight", "gp",
                      "pts", "reb", "ast", "usg_pct", "ts_pct"]
numeric_cols = [c for c in numeric_candidates if c in merged.columns]

n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.boxplot(data=merged, x="injured", y=col, ax=axes[i], palette="Set2")
    axes[i].set_title(col)
    axes[i].set_xlabel("Injured")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Feature Distributions by Injury Label", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

### 4d — Correlation Heatmap

In [ ]:
corr_cols = numeric_cols + ["injured"]
corr = merged[corr_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=ax, square=True)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## Section 5 — Feature Engineering

### 5a — Drop Non-Predictive Columns

In [ ]:
TARGET = "injured"

# Identifier and free-text columns that must not be used as features
DROP_COLS = ["player_name", "player_norm", "team_abbreviation",
             "college", "country", "season"]
drop_cols = [c for c in DROP_COLS if c in merged.columns]

feature_df = merged.drop(columns=drop_cols)
print(f"Features after dropping identifiers: {feature_df.shape[1] - 1} columns")
print("Remaining columns:", [c for c in feature_df.columns if c != TARGET])

### 5b — Encode Categorical Features

In [ ]:
cat_cols = [c for c in feature_df.select_dtypes(include=["object", "category"]).columns
            if c != TARGET]
print(f"Categorical columns to encode: {cat_cols}")

le = LabelEncoder()
for col in cat_cols:
    feature_df[col] = le.fit_transform(feature_df[col].astype(str))
    print(f"  Encoded '{col}'")

### 5c — Impute Missing Values

In [ ]:
X = feature_df.drop(columns=[TARGET])
y = feature_df[TARGET]

# Median imputation — robust to right-skewed distributions in box score stats
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

print(f"Feature matrix shape: {X_imputed.shape}")
print(f"Any remaining NaN: {X_imputed.isnull().any().any()}")
print(f"Target distribution: {y.value_counts().to_dict()}")

## Section 6 — Train / Validation / Test Split (70 / 15 / 15)

In [ ]:
# Step 1: hold out 15% as test set
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_imputed, y,
    test_size=0.15,
    stratify=y,
    random_state=RANDOM_STATE
)

# Step 2: from remaining 85%, hold out ~17.6% as validation → ~15% of total
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.15 / 0.85,
    stratify=y_trainval,
    random_state=RANDOM_STATE
)

total = len(X_imputed)
print(f"Train size:      {len(X_train):,}  ({len(X_train)/total:.1%})")
print(f"Validation size: {len(X_val):,}  ({len(X_val)/total:.1%})")
print(f"Test size:       {len(X_test):,}  ({len(X_test)/total:.1%})")
print()
for name, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name} injury rate: {yy.mean():.3f}")

## Section 7 — Random Forest Model

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=5,      # prevents overfitting on small player subgroups
    max_features="sqrt",     # standard for classification RF
    class_weight="balanced", # addresses class imbalance (fewer injured seasons)
    n_jobs=-1,
    random_state=RANDOM_STATE
)

rf.fit(X_train, y_train)
print("Training complete.")

val_proba = rf.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, val_proba)
print(f"Validation AUC-ROC: {val_auc:.4f}")

## Section 8 — Evaluation & Results

### 8a — ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_estimator(rf, X_test, y_test, ax=ax, name="Random Forest")
ax.plot([0, 1], [0, 1], "k--", label="Random baseline (AUC = 0.50)")
ax.set_title("ROC Curve — Test Set")
ax.legend()
plt.tight_layout()
plt.show()

### 8b — Classification Report

In [ ]:
y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

test_auc = roc_auc_score(y_test, y_proba)
print(f"Test AUC-ROC: {test_auc:.4f}\n")
print(classification_report(y_test, y_pred, target_names=["Not Injured", "Injured"]))

### 8c — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=["Not Injured", "Injured"])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

### 8d — Feature Importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
ax.invert_yaxis()
ax.set_title("Top 15 Feature Importances (Mean Decrease in Impurity)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

### 8e — Results Summary

| Metric | Value |
|---|---|
| Validation AUC-ROC | *(see above)* |
| Test AUC-ROC | *(see above)* |

**Dataset Join:**
- Two datasets merged on normalized player name + season string
- Players dataset provides features; injury dataset provides binary labels
- Players with no injury record in a season are labeled 0 (not injured)

**Known Limitations:**
1. **Same-season leakage:** Box score features (pts, reb, gp, etc.) are from the same season as the injury label. A stricter model would shift features back one season.
2. **Name mismatch residual:** Some players may not join due to spelling differences not caught by normalization. Fuzzy matching (e.g., `rapidfuzz`) would improve recall.
3. **Season boundary approximation:** Injuries in Oct–Dec are assigned to that year's season; edge cases near preseason may be miscategorized.
4. **Class imbalance:** Addressed with `class_weight="balanced"` in the RF. AUC-ROC is the primary metric because accuracy would be misleading.
5. **Preliminary model:** Random Forest is used here as a baseline. The final model will be an MLP with embedding layers for categorical features.